# Generate Images with the Gemini API

Generate an image from a text prompt using Google's Gemini image model through the official `google-genai` Python SDK. The image is saved as a PNG and shown in the notebook, and a small Gradio app lets you try your own prompts.

## How to run
- Open in Google Colab. Runtime type: **CPU is fine**.
- Add this secret in Colab (key icon on the left): `GEMINI_API_KEY` (create one at https://aistudio.google.com/apikey) and enable notebook access
- Run the cells from top to bottom.

_Rewritten from my original Colab notebook: `Generate image using gemeni api.ipynb`._

In [ ]:
!pip install -q -U google-genai gradio

## 1. Create the client
The API key is read from Colab Secrets, so it never appears in the notebook.

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash-image"

## 2. Generate an image
The response can contain text parts and image parts; we keep the first image.

In [ ]:
from io import BytesIO
from PIL import Image

def generate_image(prompt):
    response = client.models.generate_content(model=MODEL, contents=prompt)
    for part in response.candidates[0].content.parts:
        if part.inline_data is not None:
            return Image.open(BytesIO(part.inline_data.data))
        if part.text:
            print("Model says:", part.text)
    return None

img = generate_image("A beautiful oil painting of a peaceful mountain lake at sunrise")
if img:
    img.save("generated.png")
    display(img)

## 3. Try your own prompts

In [ ]:
import gradio as gr

gr.Interface(fn=generate_image,
             inputs=gr.Textbox(label="Prompt"),
             outputs=gr.Image(label="Result"),
             title="Gemini Image Generator").launch()